# S16 A/B testing practical — learner notebook

A fictional product team compared its current signup page **A** with a simplified page **B**. One row in the supplied synthetic CSV represents one eligible first-time visitor assigned once, and `converted` records a signup within that visitor's seven-calendar-day window. Assignment dates run from **2026-09-01 to 2026-09-14**; the last outcome window ends **2026-09-20**. The date is context, not the unit of analysis.

Earlier in S16, we learned that a displayed difference can arise from variation and that a decision also needs uncertainty, a suitable test, design checks, and a business rule. Here you will audit the visitor-level file, calculate rates and a signed effect, quantify evidence, and write a short recommendation. The CSV is fully synthetic: its planned random-assignment story is an assumption for teaching, not proof that a real platform was randomized or that a real page improved.

**Before running:** Put `S16_AB_Signup_Experiment.csv` beside this notebook. In Colab, use the Files panel to upload the CSV into the notebook's working directory. The notebook uses no website, API, generator, or network call after Python packages and the CSV are available. Tested with Python 3.12.10, NumPy 2.1.3, pandas 2.2.3, SciPy 1.14.1, statsmodels 0.14.5, and Matplotlib 3.10.7. If needed, install those tested package versions in the intended environment **before** class, then restart the kernel:

`python -m pip install numpy==2.1.3 pandas==2.2.3 scipy==1.14.1 statsmodels==0.14.5 matplotlib==3.10.7`

In Colab, use `%pip install` with the same package specifications in a separate setup cell before the practical.

## 1. Commit to the design before seeing outcomes

**Task 1 — write a short pre-result design record in this markdown cell.** Identify the eligible visitor as the unit, the control and treatment, the primary signup metric **with its assigned-visitor denominator**, and the signed comparison `p_B − p_A`. State `H0` and a **two-sided** `H1`. The planned assignment window is 14 calendar days; a conversion is measured on the assignment date or the next six days. Use the predeclared `alpha = 0.05`, Pearson chi-square on user counts with `correction=False`, a 95% Newcombe interval for B − A, and a separate **+2.0 percentage-point** minimum worthwhile improvement. What can the four-column CSV *not* prove about real assignment or logging?

**Your pre-result statement:**

> Replace this line with your design record before running the cells below.

## 2. Load and audit the input

Run the prepared cells. They check the supplied CSV without changing it. A passing file check says the rows and columns are usable; it cannot verify a real randomization platform, consistent exposure, or complete signup logging.

In [ ]:
from pathlib import Path
import hashlib
import sys

import numpy as np
import pandas as pd
import scipy
import statsmodels
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import chi2_contingency
from statsmodels.stats.proportion import confint_proportions_2indep

ALPHA = 0.05
BUSINESS_MIN_PP = 2.0
CSV_PATH = Path("S16_AB_Signup_Experiment.csv")
EXPECTED_SHA256 = "c8ce3aaff3a6eb3909b4452feb66bede9ff65a2da7017bc7dcf90e1628ea52fb"

print("Python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__)
print("SciPy", scipy.__version__, "| statsmodels", statsmodels.__version__, "| Matplotlib", matplotlib.__version__)

In [ ]:
if not CSV_PATH.is_file():
    raise FileNotFoundError(f"Place {CSV_PATH.name} beside this notebook, then rerun this cell.")
actual_sha256 = hashlib.sha256(CSV_PATH.read_bytes()).hexdigest()
if actual_sha256 != EXPECTED_SHA256:
    raise ValueError("CSV checksum differs from the supplied S16 dataset. Use the approved file unchanged.")

data = pd.read_csv(CSV_PATH)
required_columns = ["visitor_id", "variant", "converted", "assignment_date"]
assert data.columns.tolist() == required_columns, "Unexpected CSV columns or index column."
assert len(data) == 10_000, "Expected one row for each of 10,000 assigned visitors."
assert not data.isna().any().any(), "A required value is missing."
assert data["visitor_id"].is_unique, "A visitor appears more than once."
assert data["visitor_id"].str.fullmatch(r"V[0-9A-F]{16}").all(), "Invalid visitor token."
assert set(data["variant"]) == {"A", "B"}, "Only A and B are permitted."
assert pd.api.types.is_integer_dtype(data["converted"]), "converted must be integer."
assert data["converted"].isin([0, 1]).all(), "converted must be 0 or 1."

parsed_dates = pd.to_datetime(data["assignment_date"], format="%Y-%m-%d", errors="raise")
expected_dates = set(pd.date_range("2026-09-01", "2026-09-14").strftime("%Y-%m-%d"))
assert set(data["assignment_date"]) == expected_dates, "The 14 assignment dates changed."
assert parsed_dates.dt.strftime("%Y-%m-%d").eq(data["assignment_date"]).all(), "A date is not ISO YYYY-MM-DD."
assert (data.groupby("assignment_date")["variant"].nunique() == 2).all(), "Both arms must appear on every date."

allocation = data["variant"].value_counts().reindex(["A", "B"])
assert allocation.to_dict() == {"A": 5000, "B": 5000}, "Arm allocation differs from this teaching dataset."
print(f"Input checks passed: {len(data):,} unique visitor rows; {len(expected_dates)} dates; A={allocation['A']:,}, B={allocation['B']:,}.")
print("Outcome values are complete and binary. This file cannot audit real randomization, exposure, or logging accuracy.")

**Audit question:** What would a duplicate `visitor_id`, missing `converted`, or unexpected arm allocation threaten? Which real-world quality checks cannot be answered from these four columns? Write one or two sentences here before continuing.

> Your response:

## 3. Construct descriptive evidence

**Task 2:** Complete two short expressions in the next cell. The assigned count is the denominator, including every `converted = 0` visitor. Run the cell after editing; the printed rate column is in percent. A clear `NotImplementedError` means a `None` placeholder still needs your answer.

In [ ]:
# Task 2: the groupby gives the assigned denominator and converted numerator.
# Fill the two None expressions; keep nonconverters in the denominator.
summary = data.groupby("variant")["converted"].agg(assigned="size", converted="sum").reindex(["A", "B"])
summary["not_converted"] = None  # TODO: assigned minus converted
summary["rate"] = None           # TODO: converted divided by assigned
if summary[["not_converted", "rate"]].isna().any().any():
    raise NotImplementedError("Complete both Task 2 expressions before continuing.")
display(summary[["assigned", "converted", "not_converted"]].assign(conversion_rate_pct=100 * summary["rate"]))

**Task 3:** Complete the observed 2 × 2 count table and two signed effect expressions. Use A then B as rows, converted then not converted as columns. **Percentage-point change** is `100 × (B rate − A rate)`; **relative percent lift** divides the rate difference by A's rate. Check the sign and name the denominator.

In [ ]:
# Task 3: rows must be A then B; columns converted then not converted.
# Fill all three None expressions. Percentage points = 100 × (B rate − A rate).
rate_a = float(summary.loc["A", "rate"])
rate_b = float(summary.loc["B", "rate"])
observed = None          # TODO: select the 2×2 counts from summary and convert to a NumPy integer array
difference_pp = None     # TODO: signed B − A effect in percentage points
relative_lift_pct = None # TODO: relative percent lift using A as baseline
if observed is None or difference_pp is None or relative_lift_pct is None:
    raise NotImplementedError("Complete all three Task 3 expressions before continuing.")

assert observed.shape == (2, 2) and observed.sum() == len(data)
display(pd.DataFrame(observed, index=["A — control", "B — treatment"],
                     columns=["Converted (1)", "Not converted (0)"]))
print(f"A: {100*rate_a:.2f}% | B: {100*rate_b:.2f}%")
print(f"B − A: {difference_pp:+.3f} percentage points | relative lift vs A: {relative_lift_pct:+.2f}%")

**Interpretation check:** State the observed arm rates, the B − A effect in percentage points, and the relative lift using A as baseline. Why are a percentage point and a relative percent not interchangeable?

> Your response:

## 4. Evaluate null-based evidence and uncertainty

The Pearson chi-square test compares this user-count table with counts expected if variant and conversion were independent. The expected cells describe the **null model**, not future signups. Check they are comfortably above 5, and remember that independent visitor units and valid instrumentation remain assumptions. A p-value is the chance, **if the null model and assumptions hold**, of a result at least this extreme; it is not `P(H0 is true)`.

In [ ]:
# Task 4: set the approved continuity-correction choice, then run the prepared test.
correction_choice = None  # TODO: True or False? Check the pre-result rule.
if correction_choice is None:
    raise NotImplementedError("Set correction_choice for Task 4 before continuing.")
chi_result = chi2_contingency(observed, correction=correction_choice)
expected_table = pd.DataFrame(chi_result.expected_freq,
                              index=["A — control", "B — treatment"],
                              columns=["Converted (1)", "Not converted (0)"])
assert (chi_result.expected_freq >= 5).all(), "An expected cell is below 5."
display(expected_table)
print(f"Pearson chi-square = {chi_result.statistic:.6f}; df = {chi_result.dof}; two-sided p = {chi_result.pvalue:.6f}")

**Task 4 interpretation:** Compare the p-value with the predeclared `alpha = 0.05`. What do the expected converted counts mean? Can this test alone verify actual random assignment?

> Your response:

**Task 5:** The prepared call below uses a 95% **Newcombe** interval for independent proportions. Fill the two converted counts with **B first, A second**. `compare='diff'` then means `p_B − p_A`; multiply the returned probability bounds by 100 to read **percentage points**. Do not interpret a 95% frequentist interval as a 95% probability about the fixed effect.

In [ ]:
# Task 5: fill B's converted count as count1 and A's as count2.
# The assigned denominators, comparison, method, and alpha are already fixed.
count_b = None  # TODO: summary's converted count for B
count_a = None  # TODO: summary's converted count for A
if count_b is None or count_a is None:
    raise NotImplementedError("Set count_b and count_a for Task 5 before continuing.")
ci_low, ci_high = confint_proportions_2indep(
    count1=int(count_b), nobs1=int(summary.loc["B", "assigned"]),
    count2=int(count_a), nobs2=int(summary.loc["A", "assigned"]),
    compare="diff", method="newcomb", alpha=ALPHA,
)
ci_low_pp, ci_high_pp = 100 * ci_low, 100 * ci_high
print(f"95% Newcombe CI for B − A: [{ci_low_pp:+.3f}, {ci_high_pp:+.3f}] percentage points")
print(f"Predeclared minimum worthwhile gain: +{BUSINESS_MIN_PP:.1f} percentage points")

**Interval check:** Does the interval include 0? Does it include values below and above the separate +2.0 percentage-point business minimum? State what each comparison answers.

> Your response:

In [ ]:
# One observed B − A estimate with its interval; the vertical lines mark distinct rules.
fig, ax = plt.subplots(figsize=(8.2, 3.3))
ax.errorbar([difference_pp], [0],
            xerr=[[difference_pp - ci_low_pp], [ci_high_pp - difference_pp]],
            fmt="o", markersize=8, capsize=6, color="#175CD3", linewidth=2,
            label="Observed difference and 95% Newcombe CI")
ax.axvline(0, color="#667085", linestyle="--", linewidth=1.5, label="No difference (0 pp)")
ax.axvline(BUSINESS_MIN_PP, color="#B54708", linestyle=":", linewidth=2,
           label="Minimum worthwhile (+2.0 pp)")
ax.set_yticks([0], ["B − A"])
ax.set_ylim(-0.6, 0.6)
ax.set_xlim(min(-0.5, ci_low_pp - 0.35), max(3.4, ci_high_pp + 0.35))
ax.set_xlabel("Signup conversion difference (percentage points)")
ax.set_title("Observed effect, uncertainty, and business threshold")
ax.grid(axis="x", alpha=0.2)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.28), ncol=1, frameon=False)
fig.tight_layout()
display(fig)
plt.close(fig)

## 5. Write the decision note

**Your main deliverable (5–7 sentences):** Name the eligible visitor unit and primary KPI denominator; give both arm rates and the signed absolute and relative effects; report the predeclared test and B − A interval; compare the estimate and interval with +2.0 percentage points; identify one assignment/measurement or product guardrail that this file cannot check; recommend a justified next action. Do not treat `p < 0.05` as an automatic launch rule or claim that synthetic rows prove a real product effect.

> Replace this line with your note.

## 6. Optional reflection

If the true effect or valid sample size were smaller, how would detection generally change? What would a sample-ratio mismatch make you investigate before trusting an inference? This is a reasoning extension, not another test of the supplied CSV.

> Your response:

After completing your attempt, compare your work with the separately distributed trainer solution. [SciPy chi-square documentation](https://docs.scipy.org/doc/scipy-1.14.1/reference/generated/scipy.stats.chi2_contingency.html) and [statsmodels independent-proportion interval documentation](https://www.statsmodels.org/stable/generated/statsmodels.stats.proportion.confint_proportions_2indep.html) describe the two functions used here.